In [ ]:
import json
import re
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

In [ ]:
try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
    from nltk.stem import RSLPStemmer, SnowballStemmer
    NLTK_AVAILABLE = True
except Exception:
    NLTK_AVAILABLE = False

In [ ]:
ROOT_DIR = Path.cwd()
RAW_REVIEWS_PATH = ROOT_DIR / "data" / "raw" / "order_reviews.csv"
OUTPUT_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
reviews = pd.read_csv(RAW_REVIEWS_PATH)
reviews.shape
reviews.head()
reviews.info()
reviews[["review_score", "review_comment_title", "review_comment_message"]].head(10)

In [ ]:
audit = {
    "total_reviews": len(reviews),
    "reviews_with_title": reviews["review_comment_title"].notna().sum(),
    "reviews_with_message": reviews["review_comment_message"].notna().sum(),
    "low_rating_reviews": (reviews["review_score"] <= 2).sum(),
}
audit

In [ ]:
title = reviews["review_comment_title"].fillna("").astype(str)
message = reviews["review_comment_message"].fillna("").astype(str)

reviews["review_text_raw"] = (
    title + " " + message
).str.replace(r"\s+", " ", regex=True).str.strip()

In [ ]:
reviews["has_raw_text"] = reviews["review_text_raw"].str.len() > 0
reviews["is_low_rating"] = reviews["review_score"] <= 2

In [ ]:
reviews.loc[
    reviews["is_low_rating"] & reviews["has_raw_text"],
    ["review_score", "review_text_raw"]
].sample(10, random_state=42)

In [ ]:
TOKEN_PATTERN = r"[a-záàâãéêíóôõúç]+"
NORMALIZED_TOKEN_PATTERN = r"[a-z_]+"

def normalize_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    return "".join(char for char in text if not unicodedata.combining(char))

def clean_review_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[^a-záàâãéêíóôõúç0-9\s.,;:!?/-]", " ", text)
    text = re.sub(r"([!?.,;:/-])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_for_matching(text: str) -> str:
    text = normalize_accents(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_normalized_text(text: str) -> list[str]:
    return re.findall(NORMALIZED_TOKEN_PATTERN, str(text).lower())

In [ ]:
PHRASE_REPLACEMENTS = {
    r"\bainda nao recebi\b": "nao_recebi",
    r"\bnunca recebi\b": "nao_recebi",
    r"\bnao recebi\b": "nao_recebi",
    r"\bnao chegou\b": "nao_chegou",
    r"\bproduto nao chegou\b": "nao_chegou",
    r"\bpedido nao chegou\b": "nao_chegou",
    r"\bconsta entregue\b": "consta_entregue",
    r"\bmarcado como entregue\b": "marcado_entregue",

    r"\bfora do prazo\b": "fora_prazo",
    r"\bdentro do prazo\b": "dentro_prazo",
    r"\bchegou antes\b": "chegou_antes",

    r"\bproduto errado\b": "produto_errado",
    r"\bveio errado\b": "produto_errado",
    r"\bveio outro\b": "produto_errado",
    r"\bproduto diferente\b": "produto_diferente",
    r"\bveio diferente\b": "produto_diferente",
    r"\bveio faltando\b": "produto_incompleto",
    r"\bproduto incompleto\b": "produto_incompleto",
    r"\btamanho errado\b": "tamanho_errado",
    r"\bcor errada\b": "cor_errada",

    r"\bproduto quebrado\b": "produto_quebrado",
    r"\bproduto danificado\b": "produto_danificado",
    r"\bqualidade ruim\b": "qualidade_ruim",
    r"\bnao funciona\b": "nao_funciona",
    r"\bnao funcionou\b": "nao_funciona",
    r"\bmal acabado\b": "mal_acabado",

    r"\bsem resposta\b": "sem_resposta",
    r"\bsem retorno\b": "sem_resposta",
    r"\bpedi reembolso\b": "pedi_reembolso",
    r"\bpedi estorno\b": "pedi_estorno",
}

def build_phrase_text(normalized_text: str) -> str:
    text = normalized_text
    for pattern, replacement in PHRASE_REPLACEMENTS.items():
        text = re.sub(pattern, replacement, text)
    return text

In [ ]:
cleaned = reviews[["review_id", "order_id", "review_score", "review_text_raw", "is_low_rating"]].copy()
cleaned["clean_text"] = cleaned["review_text_raw"].apply(clean_review_text)
cleaned["normalized_text"] = cleaned["clean_text"].apply(normalize_for_matching)
cleaned["phrase_text"] = cleaned["normalized_text"].apply(build_phrase_text)
cleaned["token_count"] = cleaned["clean_text"].apply(lambda x: len(re.findall(TOKEN_PATTERN, x)))
cleaned["has_text"] = cleaned["token_count"] > 0

cleaned.head()

In [ ]:
FALLBACK_PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "as", "até", "com", "como", "da", "das",
    "de", "do", "dos", "e", "em", "entre", "era", "essa", "esse",
    "esta", "este", "eu", "foi", "foram", "há", "isso", "isto",
    "já", "mais", "mas", "me", "meu", "minha", "muito", "na",
    "nas", "no", "nos", "o", "os", "ou", "para", "pela", "pelo",
    "por", "qual", "quando", "que", "quem", "se", "seu", "sua",
    "também", "tem", "ter", "um", "uma", "você", "vocês"
}

PROTECTED_NEGATIONS = {"não", "nao", "nunca", "nem", "sem"}

DOMAIN_STOPWORDS = {
    "lannister", "stark", "targaryen", "baratheon"
}

TOPIC_EXTRA_STOPWORDS = {
    "nao", "produto", "pedido", "compra", "comprei",
    "recebi", "veio", "ainda", "dia", "dias", "loja",
    "entrega", "comprado", "comprar"
}

def load_library_portuguese_stopwords() -> set[str]:
    stopwords = set(FALLBACK_PORTUGUESE_STOPWORDS)

    try:
        from nltk.corpus import stopwords as nltk_stopwords
        stopwords.update(nltk_stopwords.words("portuguese"))
        print("Using NLTK Portuguese stopwords.")
    except Exception as exc:
        warnings.warn(f"NLTK Portuguese stopwords unavailable. Using fallback. Reason: {exc}")

    stopwords = {normalize_for_matching(word) for word in stopwords}
    protected = {normalize_for_matching(word) for word in PROTECTED_NEGATIONS}

    matching_stopwords = (stopwords | DOMAIN_STOPWORDS) - protected
    topic_stopwords = matching_stopwords | TOPIC_EXTRA_STOPWORDS

    return matching_stopwords, topic_stopwords

matching_stopwords, topic_stopwords = load_library_portuguese_stopwords()
len(matching_stopwords), len(topic_stopwords)

In [ ]:
sample_cols = ["review_score", "review_text_raw", "clean_text", "normalized_text", "phrase_text"]
cleaned.loc[cleaned["is_low_rating"] & cleaned["has_text"], sample_cols].sample(10, random_state=42)

In [ ]:
THEME_PATTERNS = {
    "theme_delivery_not_received": [
        "nao_recebi", "nao_chegou", "aguardando",
        "consta_entregue", "marcado_entregue"
    ],
    "theme_delivery_delay": [
        "atraso", "atrasado", "atrasada", "atrasou",
        "demora", "demorou", "fora_prazo",
        "prazo", "rastreio", "transportadora", "correios"
    ],
    "theme_product_wrong_incomplete": [
        "produto_errado", "produto_diferente", "produto_incompleto",
        "faltou", "faltando", "incompleto", "diferente",
        "tamanho_errado", "cor_errada"
    ],
    "theme_product_defect_quality": [
        "defeito", "defeituoso", "defeituosa",
        "quebrado", "quebrada", "quebrados",
        "danificado", "danificada", "danificados",
        "qualidade_ruim", "pessimo", "ruim",
        "nao_funciona", "mal_acabado"
    ],
    "theme_seller_service_refund": [
        "reembolso", "estorno", "devolucao",
        "pedi_reembolso", "pedi_estorno",
        "atendimento", "sem_resposta",
        "contato", "vendedor", "reclamei", "troca"
    ],
    "theme_positive_recommendation": [
        "recomendo", "gostei", "otimo", "excelente",
        "bom", "perfeito", "chegou_antes", "dentro_prazo"
    ],
}

In [ ]:
THEME_LABELS = {
    "theme_delivery_not_received": "Delivery - Not Received",
    "theme_delivery_delay": "Delivery - Delay",
    "theme_product_wrong_incomplete": "Product - Wrong/Incomplete",
    "theme_product_defect_quality": "Product - Defect/Quality",
    "theme_seller_service_refund": "Seller/Service/Refund",
    "theme_positive_recommendation": "Positive Recommendation",
    "theme_other": "Other Complaint",
}

def tag_review_themes(df: pd.DataFrame) -> pd.DataFrame:
    tags = df[["review_id", "order_id", "review_score", "phrase_text", "has_text", "is_low_rating"]].copy()

    for theme, patterns in THEME_PATTERNS.items():
        tags[theme] = tags["phrase_text"].apply(
            lambda text: any(pattern in text.split() or pattern in text for pattern in patterns)
        )

    theme_cols = list(THEME_PATTERNS.keys())
    tags["theme_count_without_other"] = tags[theme_cols].sum(axis=1).astype(int)
    tags["theme_other"] = tags["theme_count_without_other"] == 0
    tags["theme_count"] = tags[theme_cols + ["theme_other"]].sum(axis=1).astype(int)

    return tags

tags = tag_review_themes(cleaned)
tags.head()

In [ ]:
def summarize_themes(tags: pd.DataFrame) -> pd.DataFrame:
    theme_columns = list(THEME_PATTERNS.keys()) + ["theme_other"]
    low_rating = tags[(tags["is_low_rating"]) & (tags["has_text"])].copy()
    denominator = max(len(low_rating), 1)

    rows = []
    for theme in theme_columns:
        matched = low_rating[low_rating[theme]]
        rows.append({
            "theme": THEME_LABELS[theme],
            "review_count": int(len(matched)),
            "percentage_of_low_rating_reviews": round(len(matched) * 100.0 / denominator, 2),
            "avg_review_score": round(float(matched["review_score"].mean()), 2) if len(matched) else None,
            "sample_keywords": ", ".join(THEME_PATTERNS.get(theme, ["no matched theme"])[:8]),
        })

    return pd.DataFrame(rows).sort_values("review_count", ascending=False)

theme_summary = summarize_themes(tags)
theme_summary

In [ ]:
def vectorizer_tokenizer(text: str) -> list[str]:
    return tokenize_normalized_text(text)

def make_tfidf_vectorizer(stopwords: set[str], min_df=5, max_df=0.8, max_features=3000):
    return TfidfVectorizer(
        tokenizer=vectorizer_tokenizer,
        token_pattern=None,
        stop_words=sorted(stopwords),
        ngram_range=(1, 2),
        min_df=min_df,
        max_df=max_df,
        max_features=max_features,
    )

In [ ]:
text_reviews = cleaned[cleaned["has_text"]].copy()

segments = {
    "review_score_1": text_reviews[text_reviews["review_score"] == 1],
    "review_score_2": text_reviews[text_reviews["review_score"] == 2],
    "low_rating_score_1_2": text_reviews[text_reviews["review_score"] <= 2],
    "high_rating_score_4_5": text_reviews[text_reviews["review_score"] >= 4],
    "all_text_reviews": text_reviews,
}

In [ ]:
def get_top_terms(segment_name, texts, stopwords, top_n=30):
    if len(texts) == 0:
        return []

    vectorizer = make_tfidf_vectorizer(stopwords)
    matrix = vectorizer.fit_transform(texts)

    terms = np.array(vectorizer.get_feature_names_out())
    mean_tfidf = np.asarray(matrix.mean(axis=0)).ravel()

    count_vectorizer = CountVectorizer(
        tokenizer=vectorizer_tokenizer,
        token_pattern=None,
        stop_words=sorted(stopwords),
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.8,
        max_features=3000,
    )
    count_matrix = count_vectorizer.fit_transform(texts)
    count_terms = np.array(count_vectorizer.get_feature_names_out())
    raw_counts = np.asarray(count_matrix.sum(axis=0)).ravel()
    count_lookup = dict(zip(count_terms, raw_counts))

    top_indices = mean_tfidf.argsort()[::-1][:top_n]

    rows = []
    for idx in top_indices:
        term = terms[idx]
        rows.append({
            "segment": segment_name,
            "term": term,
            "score_or_weight": round(float(mean_tfidf[idx]), 6),
            "raw_count": int(count_lookup.get(term, 0)),
        })

    return rows

rows = []
for segment_name, segment_df in segments.items():
    rows.extend(get_top_terms(segment_name, segment_df["phrase_text"], matching_stopwords))

top_terms = pd.DataFrame(rows)
top_terms[top_terms["segment"] == "low_rating_score_1_2"].head(30)

In [ ]:
def label_topic(top_terms: list[str]) -> str:
    joined = " ".join(top_terms)

    scores = {}
    for theme, patterns in THEME_PATTERNS.items():
        scores[theme] = sum(1 for pattern in patterns if pattern in joined)

    best_theme, score = max(scores.items(), key=lambda x: x[1])
    if score == 0:
        return "Other Complaint"
    return THEME_LABELS[best_theme]

def build_nmf_topics(df: pd.DataFrame, stopwords: set[str], n_topics=6):
    low_text = df[(df["is_low_rating"]) & (df["has_text"])].copy()

    vectorizer = TfidfVectorizer(
        tokenizer=vectorizer_tokenizer,
        token_pattern=None,
        stop_words=sorted(stopwords),
        ngram_range=(1, 2),
        min_df=10,
        max_df=0.6,
        max_features=2000,
    )

    matrix = vectorizer.fit_transform(low_text["phrase_text"])

    nmf = NMF(
        n_components=n_topics,
        random_state=42,
        init="nndsvda",
        max_iter=500,
    )

    doc_topics = nmf.fit_transform(matrix)
    dominant_topics = doc_topics.argmax(axis=1)

    terms = np.array(vectorizer.get_feature_names_out())
    topic_counts = Counter(dominant_topics)

    rows = []
    for topic_id, weights in enumerate(nmf.components_):
        top_terms = terms[weights.argsort()[::-1][:12]].tolist()
        rows.append({
            "topic_id": int(topic_id),
            "top_terms": ", ".join(top_terms),
            "manual_topic_label": label_topic(top_terms),
            "review_count": int(topic_counts.get(topic_id, 0)),
        })

    return pd.DataFrame(rows).sort_values("review_count", ascending=False)

nmf_topics = build_nmf_topics(cleaned, topic_stopwords, n_topics=6)
nmf_topics

In [ ]:
def build_lda_topics(df: pd.DataFrame, stopwords: set[str], n_topics=6):
    low_text = df[(df["is_low_rating"]) & (df["has_text"])].copy()

    vectorizer = CountVectorizer(
        tokenizer=vectorizer_tokenizer,
        token_pattern=None,
        stop_words=sorted(stopwords),
        ngram_range=(1, 2),
        min_df=10,
        max_df=0.6,
        max_features=2000,
    )

    matrix = vectorizer.fit_transform(low_text["phrase_text"])

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method="batch",
        max_iter=30,
    )

    doc_topics = lda.fit_transform(matrix)
    dominant_topics = doc_topics.argmax(axis=1)

    terms = np.array(vectorizer.get_feature_names_out())
    topic_counts = Counter(dominant_topics)

    rows = []
    for topic_id, weights in enumerate(lda.components_):
        top_terms = terms[weights.argsort()[::-1][:12]].tolist()
        rows.append({
            "topic_id": int(topic_id),
            "top_terms": ", ".join(top_terms),
            "manual_topic_label": label_topic(top_terms),
            "review_count": int(topic_counts.get(topic_id, 0)),
        })

    return pd.DataFrame(rows).sort_values("review_count", ascending=False)

lda_topics = build_lda_topics(cleaned, topic_stopwords, n_topics=6)
lda_topics

In [ ]:
def sample_reviews_for_theme(theme_col, n=10):
    ids = tags[(tags[theme_col]) & (tags["is_low_rating"]) & (tags["has_text"])]["review_id"]
    return cleaned[cleaned["review_id"].isin(ids)][
        ["review_score", "review_text_raw", "phrase_text"]
    ].sample(min(n, len(ids)), random_state=42)

sample_reviews_for_theme("theme_delivery_not_received", 10)
sample_reviews_for_theme("theme_product_defect_quality", 10)
sample_reviews_for_theme("theme_seller_service_refund", 10)

In [ ]:
cleaned_output = cleaned[
    [
        "review_id", "order_id", "review_score",
        "review_text_raw", "clean_text", "normalized_text",
        "phrase_text", "token_count", "has_text", "is_low_rating"
    ]
]

theme_tags_output = tags.drop(columns=["phrase_text"], errors="ignore")

cleaned_output.to_csv(OUTPUT_DIR / "review_text_cleaned.csv", index=False)
theme_tags_output.to_csv(OUTPUT_DIR / "review_theme_tags.csv", index=False)
theme_summary.to_csv(OUTPUT_DIR / "review_theme_summary.csv", index=False)
top_terms.to_csv(OUTPUT_DIR / "review_top_terms_by_score.csv", index=False)
nmf_topics.to_csv(OUTPUT_DIR / "review_topics_nmf.csv", index=False)
lda_topics.to_csv(OUTPUT_DIR / "review_topics_lda.csv", index=False)

audit = {
    "total_reviews": int(len(cleaned)),
    "reviews_with_any_text": int(cleaned["has_text"].sum()),
    "low_rating_reviews": int(cleaned["is_low_rating"].sum()),
    "low_rating_reviews_with_text": int((cleaned["is_low_rating"] & cleaned["has_text"]).sum()),
    "percentage_reviews_with_text": round(float(cleaned["has_text"].mean() * 100), 2),
    "percentage_low_rating_with_text": round(
        float((cleaned["is_low_rating"] & cleaned["has_text"]).sum() * 100 / max(cleaned["is_low_rating"].sum(), 1)),
        2
    ),
    "median_token_count_all_text": float(cleaned.loc[cleaned["has_text"], "token_count"].median()),
    "median_token_count_low_rating_text": float(cleaned.loc[cleaned["is_low_rating"] & cleaned["has_text"], "token_count"].median()),
}

(OUTPUT_DIR / "review_nlp_audit.json").write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

audit